# 87. Scikit-learn工作流与数据切分

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 2 / 34 步：建立训练、切分与预处理工作流**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 机器学习模块入门  →  **本章任务：** Scikit-learn工作流与数据切分  →  **下一步：** 数据预处理与Pipeline
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：在真实数据分析中，模型的作用不是把训练数据的分数再报一遍，而是要对没见过的数据做出可靠预测。scikit-learn 把「训练、预测、评估」这一整套流程封装成统一接口，让你把精力放在数据理解和结果判断上。下面先从最核心的三个动作入手，再动手训练你的第一个模型。




## 本章目标

学完本章，你将能够：

- **理解**：理解「Scikit-learn工作流与数据切分」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「Scikit-learn工作流与数据切分」的关键输出指标。
- **迁移**：能把「Scikit-learn工作流与数据切分」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 87.1 核心概念

在真实数据分析中，模型的作用不是把训练数据的分数再报一遍，而是要对没见过的数据做出可靠预测。scikit-learn 把「训练、预测、评估」这一整套流程封装成统一接口，让你把精力放在数据理解和结果判断上。下面先从最核心的三个动作入手，再动手训练你的第一个模型。

- fit 只在训练集学习参数
- predict 在未参与训练的数据上产生结果
- 测试集不能参与模型选择（打个比方：训练集像“刷练习册”，测试集像“正式考试”——练习刷得再熟不算数，真正本事要用没见过的正式卷来验。）
- 固定 random_state 便于复现，不代表结果天然稳定


## 87.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 认识 sklearn 数据集 | `pd.concat()`、`X.head()`、`y.head()`、`.rename()` | Iris 是 150 行、4 个数值特征、3 个类别的经典多分类数据。 | 在全部数据上训练后再报告同一数据的得分 |
| 切分、训练与基线 | `model.predict()`、`dummy.predict()`、`pd.DataFrame()`、`y_train.value_counts()` | 分层切分后训练逻辑回归，并与只预测多数类的模型比较。 | 反复查看测试集并据此调参 |


## 87.3 示例 1：认识 sklearn 数据集

Iris 是 150 行、4 个数值特征、3 个类别的经典多分类数据。


<!-- math-foundation:chapter-87 -->
### 数学推导｜机器学习是在未见数据上最小化风险

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜先定义单个样本的损失。** 预测函数 $f$ 在样本 $(x_i,y_i)$ 上产生 $L_i=L(y_i,f(x_i))$。

**第 2 步｜真正关心的是未来总体风险。** 若未来数据来自分布 $P$，理想目标是

$$
R(f)=\mathbb{E}_{(X,Y)\sim P}[L(Y,f(X))]
$$

**第 3 步｜用训练样本近似未知期望。** 经验风险 $\hat R(f)=\sum_iL_i/n$ 是可计算代理。模型在训练集最小化它，验证集估计方案选择后的泛化表现，测试集只做最终审计。

**把上面的关系收束为本章计算式：**

$$
\hat{R}(f)=\frac{1}{n}\sum_{i=1}^{n}L\bigl(y_i,f(x_i)\bigr)
$$

**符号解释：** $L$ 是损失函数，$\hat{R}$ 是样本上的经验风险。

**代码对应：** 训练只使用训练集拟合；验证集选方案；测试集只做最终一次评估。

**使用边界：** 训练误差低不代表泛化好；数据泄漏会让评估虚高。


In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
X, y = iris.data, iris.target
print("特征形状:", X.shape, "目标形状:", y.shape)
print("类别:", dict(enumerate(iris.target_names)))
display(pd.concat([X.head(), y.head().rename("target")], axis=1))


**练一练**：把鸢尾花数据中 `petal length (cm)` 这一列整体加 2.0，得到新数据 `X_modified`，并对比加值前后该列的前 3 个值。观察：是只有这一列变化，还是全部列都变了？用一句话写出你的观察。


In [ ]:
# 请在下方填写代码
# 1. 从 X 复制出一份新数据 X_modified
# 2. 把 X_modified 的 "petal length (cm)" 列整体加 2.0
# 3. 打印加值前后该列各自的前 3 个值（用 .head(3).values）
# TODO：请在下方完成 —— 练一练：把鸢尾花数据中 petal length (cm) 这一列整体加 2.0，得到新数据 X_modified，并对


In [ ]:
# 请在下方填写代码
X_modified = X.copy()
X_modified["petal length (cm)"] = X_modified["petal length (cm)"] + 2.0

before = X["petal length (cm)"].head(3).values
after = X_modified["petal length (cm)"].head(3).values
print("加值前前3个:", before)
print("加值后前3个:", after)
print("差值:", after - before)


## 87.4 示例 2：切分、训练与基线

分层切分后训练逻辑回归，并与只预测多数类的模型比较。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=76
)
model = LogisticRegression(max_iter=500).fit(X_train, y_train)
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("模型准确率:", round(accuracy_score(y_test, model.predict(X_test)), 3))
print("多数类基线:", round(accuracy_score(y_test, dummy.predict(X_test)), 3))
print("训练/测试类别比例:")
print(
    pd.DataFrame(
        {
            "train": y_train.value_counts(normalize=True),
            "test": y_test.value_counts(normalize=True),
        }
    ).round(3)
)


## 87.5 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 87.6 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 87.7 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 87.7.1 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 87.7.2 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 87.8 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 87.8.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 87.9 易错点提醒

- 在全部数据上训练后再报告同一数据的得分
- 反复查看测试集并据此调参
- 类别不平衡时忘记 stratify
- 没有任何简单基线就宣称模型有效


## 87.10 练习与作业

1. 把 test_size 改为 0.3
2. 训练一个 StandardScaler + LogisticRegression 流水线
3. 比较新模型与多数类基线

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 87.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“把 test_size 改为 0.3”。
2. **独立完成**：不复制示例代码，完成“训练一个 StandardScaler + LogisticRegression 流水线”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“比较新模型与多数类基线”，用一两句话说明你修改了什么。

### 87.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 87.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris

# 重新载入 iris 数据，避免被上方实训实验覆盖的 X/y 影响本练习
iris = load_iris(as_frame=True)
X, y = iris.data, iris.target

Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=76
)
practice_model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=500)
).fit(Xp_train, yp_train)
practice_score = practice_model.score(Xp_test, yp_test)
print("练习准确率:", round(practice_score, 3))


## 87.12 小结

建立 scikit-learn 的标准监督学习工作流：准备 X/y、分层切分、拟合模型、预测并与简单基线比较。


### 87.12.1 你已经掌握

- 区分特征矩阵 X 与目标 y
- 使用训练集和测试集评估泛化能力
- 分类任务使用 stratify 保持类别比例
- 用 DummyClassifier 建立最低可接受基线


### 87.12.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 87.12.3 需要注意

- 在全部数据上训练后再报告同一数据的得分
- 反复查看测试集并据此调参
- 类别不平衡时忘记 stratify
- 没有任何简单基线就宣称模型有效


### 87.12.4 完成检查

- [ ] 能够区分特征矩阵 X 与目标 y
- [ ] 能够使用训练集和测试集评估泛化能力
- [ ] 能够分类任务使用 stratify 保持类别比例
- [ ] 能够用 DummyClassifier 建立最低可接受基线


### 87.12.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
